In [21]:
import random
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.sparse.linalg import eigsh
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from torch_geometric.data import Data
from torch_geometric.nn import GINConv
from torch_geometric.nn.norm import LayerNorm
from torch_geometric.utils import dropout_edge

In [37]:
import sys

try:
    REPO_ROOT = Path(__file__).resolve().parent
except NameError:
    REPO_ROOT = Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from Models.GraphGPS import GraphGPSRegressor

In [22]:
DATA_FILE = "merged_file_new_2.csv"  # путь к твоему CSV
ID_COL = "vk_id"
TARGETS = [
    "result/0/Экстраверсия–интроверсия",
    "result/0/Привязанность–обособленность",
    "result/0/Самоконтроль–импульсивность",
    "result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость",
    "result/0/Экспрессивность–практичность"
]

In [23]:
TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAINVAL = 0.25  # 0.25 -> итоговое разбиение 60/20/20
RANDOM_SEED = 42
K_NEIGHBORS = 10
PE_DIM = 4
EPOCHS = 300
EARLY_STOPPING_PATIENCE = 30
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 32
EMBED_DIM = 32
NUM_LAYERS = 2
HEADS = 2
DROPOUT = 0.2
DROPEDGE_P = 0.5
OUTPUT_DIR = REPO_ROOT / "local_experiment_outputs"

In [24]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_header(text: str) -> None:
    print("\n" + "=" * 80)
    print(text)
    print("=" * 80)

In [31]:
import csv
from pathlib import Path
import pandas as pd


def read_table_robust(path: str | Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")

    suffix = path.suffix.lower()

    # Excel
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    # Parquet
    if suffix == ".parquet":
        return pd.read_parquet(path)

    # CSV / TXT
    read_attempts = [
        # обычные варианты
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": "\t", "encoding": "utf-8"},
        {"sep": ",", "encoding": "utf-8-sig"},
        {"sep": ";", "encoding": "utf-8-sig"},
        {"sep": ",", "encoding": "cp1251"},
        {"sep": ";", "encoding": "cp1251"},
        {"sep": "\t", "encoding": "cp1251"},
    ]

    errors = []

    for params in read_attempts:
        try:
            df = pd.read_csv(
                path,
                sep=params["sep"],
                encoding=params["encoding"],
                engine="python",          # устойчивее C engine
                on_bad_lines="warn",      # плохие строки не валят всё
            )
            if df.shape[1] > 1:
                print(f"Файл прочитан как CSV: sep={repr(params['sep'])}, encoding={params['encoding']}")
                return df
        except Exception as e:
            errors.append(f"{params}: {repr(e)}")

    # Последняя попытка: игнорировать кавычки вообще
    try:
        df = pd.read_csv(
            path,
            sep=None,                    # автоопределение разделителя
            encoding="utf-8",
            engine="python",
            on_bad_lines="warn",
            quoting=csv.QUOTE_NONE,
        )
        if df.shape[1] > 1:
            print("Файл прочитан в аварийном режиме (QUOTE_NONE).")
            return df
    except Exception as e:
        errors.append(f"QUOTE_NONE fallback: {repr(e)}")

    raise ValueError(
        "Не удалось прочитать файл.\n"
        "Попытки чтения:\n" + "\n".join(errors[:20])
    )


def load_single_file_dataset(data_file: str, id_col: str, targets: list[str]):
    df_all = read_table_robust(data_file)

    if id_col not in df_all.columns:
        raise ValueError(
            f"В файле нет колонки '{id_col}'.\n"
            f"Найденные колонки:\n{list(df_all.columns)}"
        )

    missing_targets = [c for c in targets if c not in df_all.columns]
    if missing_targets:
        raise ValueError(
            f"Не найдены target-колонки: {missing_targets}\n"
            f"Найденные колонки:\n{list(df_all.columns)}"
        )

    # Убираем полностью пустые строки
    df_all = df_all.dropna(how="all").copy()

    # Убираем дубли по id
    df_all = df_all.drop_duplicates(subset=[id_col]).copy()

    # target-таблица
    df_targets = df_all[[id_col] + targets].copy()

    # feature-таблица
    feature_cols = [c for c in df_all.columns if c not in [id_col] + targets]
    df_features_raw = df_all[[id_col] + feature_cols].copy()

    return df_features_raw, df_targets, feature_cols

def preprocess_features(df_features_raw: pd.DataFrame, id_col: str) -> pd.DataFrame:
    df_work = df_features_raw.copy()
    feature_cols = [c for c in df_work.columns if c != id_col]

    if not feature_cols:
        raise ValueError("После исключения id-колонки не осталось признаков.")

    # 1) Убираем признаки, которые полностью пустые
    all_nan_cols = [c for c in feature_cols if df_work[c].isna().all()]
    if all_nan_cols:
        print(f"Удаляю полностью пустые признаки: {len(all_nan_cols)}")
        df_work = df_work.drop(columns=all_nan_cols)
        feature_cols = [c for c in feature_cols if c not in all_nan_cols]

    if not feature_cols:
        raise ValueError("После удаления пустых колонок не осталось признаков.")

    # 2) Делим на числовые и категориальные
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_work[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    transformers = []

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_cols,
        ))

    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_cols,
        ))

    if not transformers:
        raise ValueError("Не удалось определить ни числовые, ни категориальные признаки.")

    # 3) Преобразуем
    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    X_arr = preprocessor.fit_transform(df_work[feature_cols])

    # 4) Берём имена признаков прямо из ColumnTransformer,
    #    а не собираем вручную — так размеры всегда совпадут
    out_cols = preprocessor.get_feature_names_out().tolist()

    # Чистим префиксы num__ / cat__
    cleaned_cols = []
    for c in out_cols:
        if "__" in c:
            cleaned_cols.append(c.split("__", 1)[1])
        else:
            cleaned_cols.append(c)

    # 5) Собираем DataFrame
    df_features = pd.DataFrame(X_arr, columns=cleaned_cols, index=df_work.index)
    df_features.insert(0, id_col, df_work[id_col].values)

    # 6) Мягкая стабилизация сильно скошенных неотрицательных признаков
    feature_only_cols = [c for c in df_features.columns if c != id_col]
    for col in feature_only_cols:
        if pd.api.types.is_numeric_dtype(df_features[col]):
            col_min = df_features[col].min()
            if pd.notna(col_min) and col_min >= 0:
                skew = pd.Series(df_features[col]).skew()
                if pd.notna(skew) and abs(skew) > 1.0:
                    df_features[col] = np.log1p(df_features[col])

    return df_features

In [32]:
def build_knn_edges(df_nodes: pd.DataFrame, id_col: str = "vk_id", k: int = 10) -> pd.DataFrame:
    if len(df_nodes) < 2:
        raise ValueError("Для построения графа нужно минимум 2 объекта.")

    X_num = df_nodes.drop(columns=[id_col]).select_dtypes(include=[np.number]).copy().fillna(0)
    if X_num.shape[1] == 0:
        raise ValueError("После препроцессинга не осталось числовых признаков для построения графа.")

    ids = df_nodes[id_col].to_numpy()
    n_neighbors = min(k + 1, len(df_nodes))

    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(X_num)
    _, indices = nn.kneighbors(X_num)

    edges = set()
    for i, neighs in enumerate(indices):
        for j in neighs[1:]:  # первый сосед — сам объект
            a, b = ids[i], ids[j]
            if a == b:
                continue
            pair = tuple(sorted((a, b)))
            edges.add(pair)

    edges_df = pd.DataFrame(sorted(edges), columns=["source", "target"])

    # fallback: если граф пустой, строим простую цепочку
    if edges_df.empty:
        chain_edges = [(ids[i], ids[i + 1]) for i in range(len(ids) - 1)]
        edges_df = pd.DataFrame(chain_edges, columns=["source", "target"])

    return edges_df


In [33]:
class GPSLayer(nn.Module):
    def __init__(self, dim: int, heads: int = 2, dropout: float = 0.5):
        super().__init__()

        self.local_mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ELU(),
            nn.Linear(dim, dim),
        )

        self.local_conv = GINConv(nn=self.local_mlp, train_eps=True)
        self.local_norm = LayerNorm(dim)
        self.local_dropout = nn.Dropout(dropout)

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_norm = LayerNorm(dim)
        self.attn_dropout = nn.Dropout(dropout)

        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
        )
        self.ffn_norm = LayerNorm(dim)
        self.ffn_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        residual = x

        x_local = self.local_conv(x, edge_index)
        x_local = self.local_norm(x_local)
        x_local = F.elu(x_local)
        x_local = self.local_dropout(x_local)
        x = residual + x_local

        x_global = x.unsqueeze(0)
        attn_out, _ = self.attn(x_global, x_global, x_global, need_weights=False)
        attn_out = attn_out.squeeze(0)
        attn_out = self.attn_norm(attn_out)
        attn_out = self.attn_dropout(attn_out)
        x = x + attn_out

        x_ffn = self.ffn(x)
        x_ffn = self.ffn_norm(x_ffn)
        x_ffn = self.ffn_dropout(x_ffn)
        x = x + x_ffn
        return x


class GraphGPSRegressor(nn.Module):
    def __init__(
            self,
            node_dim: int,
            hidden_dim: int = 32,
            embed_dim: int = 32,
            num_targets: int = 5,
            num_layers: int = 2,
            pe_dim: int = 4,
            heads: int = 2,
            dropout: float = 0.5,
            dropedge_p: float = 0.2,
    ):
        super().__init__()

        self.pe_dim = pe_dim
        self.dropedge_p = dropedge_p

        self.input_proj = nn.Sequential(
            nn.Linear(node_dim + pe_dim, hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout),
        )

        self.layers = nn.ModuleList([
            GPSLayer(dim=hidden_dim, heads=heads, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, embed_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_targets),
        )

    def add_noise(self, x: torch.Tensor, noise_std: float = 0.02) -> torch.Tensor:
        if self.training:
            return x + torch.randn_like(x) * noise_std
        return x

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.add_noise(x, noise_std=0.02)

        if self.training and self.dropedge_p > 0:
            edge_index, _ = dropout_edge(edge_index=edge_index, p=self.dropedge_p)

        for layer in self.layers:
            x = layer(x, edge_index)

        out = self.head(x)
        return out


class PsychologyGraph:
    def __init__(self, df_nodes: pd.DataFrame, df_edges: pd.DataFrame, targets: list[str], id_col: str = "vk_id"):
        self.df_nodes = df_nodes.reset_index(drop=True).copy()
        self.df_edges = df_edges.copy()
        self.targets_cols = targets
        self.id_col = id_col

        self.user_ids = self.df_nodes[id_col].values
        exclude_cols = [id_col] + targets
        self.node_feature_cols = [c for c in self.df_nodes.columns if c not in exclude_cols]

        self.node_features = self.df_nodes[self.node_feature_cols].values.astype(np.float32)
        n_quantiles_features = max(10, min(1000, len(self.df_nodes)))
        self.node_scaler = QuantileTransformer(n_quantiles=n_quantiles_features, random_state=RANDOM_SEED)
        self.node_features_norm = self.node_scaler.fit_transform(self.node_features)

        self.targets = self.df_nodes[self.targets_cols].values.astype(np.float32)
        n_quantiles_targets = max(10, min(1000, len(self.df_nodes)))
        self.target_scaler = QuantileTransformer(n_quantiles=n_quantiles_targets, random_state=RANDOM_SEED)
        self.targets_norm = self.target_scaler.fit_transform(self.targets)

        self.id_to_idx = {uid: i for i, uid in enumerate(self.user_ids)}

    def build_graph(self) -> nx.Graph:
        G = nx.Graph()
        for uid in self.user_ids:
            G.add_node(uid)

        for _, row in self.df_edges.iterrows():
            u = row["source"]
            v = row["target"]
            if u in G and v in G and u != v:
                G.add_edge(u, v)

        for i, uid in enumerate(self.user_ids):
            G.nodes[uid]["features"] = self.node_features_norm[i]

        return G

    def get_laplacian_pe_from_graph(self, G: nx.Graph, k: int = 4) -> torch.Tensor:
        num_nodes = len(G.nodes())
        if num_nodes <= 2 or G.number_of_edges() == 0:
            return torch.zeros((num_nodes, k), dtype=torch.float32)

        try:
            L = nx.normalized_laplacian_matrix(G).astype(np.float64)
            eig_k = min(k + 1, max(1, num_nodes - 1))
            eig_vals, eig_vecs = eigsh(L, k=eig_k, which="SM")
            idx = eig_vals.argsort()
            eig_vecs = eig_vecs[:, idx]

            if eig_vecs.shape[1] <= 1:
                pe = np.zeros((num_nodes, k), dtype=np.float32)
            else:
                pe = eig_vecs[:, 1 : min(k + 1, eig_vecs.shape[1])]
                if pe.shape[1] < k:
                    pad = np.zeros((num_nodes, k - pe.shape[1]), dtype=np.float64)
                    pe = np.concatenate([pe, pad], axis=1)
                pe = (pe - pe.mean(axis=0)) / (pe.std(axis=0) + 1e-8)
            return torch.tensor(pe, dtype=torch.float32)
        except Exception as e:
            print(f"Warning: PE computation failed: {e}")
            return torch.zeros((num_nodes, k), dtype=torch.float32)

    def to_pyg_data(self, G: nx.Graph, add_pe: bool = True, pe_dim: int = 0) -> Data:
        node_list = list(G.nodes())
        node_to_idx = {node: idx for idx, node in enumerate(node_list)}

        x = torch.tensor(
            np.stack([G.nodes[node]["features"] for node in node_list]),
            dtype=torch.float32,
        )

        if add_pe and pe_dim > 0:
            pe = self.get_laplacian_pe_from_graph(G, k=pe_dim)
            x = torch.cat([x, pe], dim=-1)

        edges = [[node_to_idx[u], node_to_idx[v]] for u, v in G.edges()]
        if len(edges) == 0:
            # fallback: self-loop chain if graph degenerated
            edges = [[i, i] for i in range(len(node_list))]

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

        # Делаем граф неориентированным в формате edge_index
        rev_edge_index = edge_index[[1, 0], :]
        edge_index = torch.cat([edge_index, rev_edge_index], dim=1)

        y = torch.tensor(
            np.stack([self.targets_norm[self.id_to_idx[node]] for node in node_list]),
            dtype=torch.float32,
        )

        return Data(x=x, edge_index=edge_index, y=y)

    def inverse_transform_targets(self, y_pred_norm: np.ndarray | torch.Tensor) -> np.ndarray:
        if isinstance(y_pred_norm, torch.Tensor):
            y_pred_norm = y_pred_norm.detach().cpu().numpy()
        return self.target_scaler.inverse_transform(y_pred_norm)

In [34]:
def add_masks(data: Data, seed: int = 42) -> Data:
    num_nodes = data.num_nodes
    all_indices = np.arange(num_nodes)

    train_val_idx, test_idx = train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=seed,
        shuffle=True,
    )
    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=VAL_SIZE_FROM_TRAINVAL,
        random_state=seed,
        shuffle=True,
    )

    data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    data.train_mask[torch.tensor(train_idx, dtype=torch.long)] = True
    data.val_mask[torch.tensor(val_idx, dtype=torch.long)] = True
    data.test_mask[torch.tensor(test_idx, dtype=torch.long)] = True
    return data


def train_epoch(model: nn.Module, data: Data, criterion, optimizer, device: torch.device) -> float:
    model.train()
    optimizer.zero_grad()

    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    y = data.y.to(device)

    out = model(x, edge_index)
    loss = criterion(out[data.train_mask], y[data.train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return float(loss.item())


def evaluate_split(model: nn.Module, data: Data, mask_name: str, criterion, device: torch.device, graph_obj: PsychologyGraph):
    model.eval()
    mask = getattr(data, mask_name)

    with torch.no_grad():
        x = data.x.to(device)
        edge_index = data.edge_index.to(device)
        y = data.y.to(device)
        out = model(x, edge_index)

        y_true_norm = y[mask]
        y_pred_norm = out[mask]
        loss = criterion(y_pred_norm, y_true_norm).item()

    y_true_real = graph_obj.inverse_transform_targets(y_true_norm)
    y_pred_real = graph_obj.inverse_transform_targets(y_pred_norm)

    per_target = pd.DataFrame({
        "Target": graph_obj.targets_cols,
        "MAE": mean_absolute_error(y_true_real, y_pred_real, multioutput="raw_values"),
        "RMSE": np.sqrt(mean_squared_error(y_true_real, y_pred_real, multioutput="raw_values")),
        "R2": r2_score(y_true_real, y_pred_real, multioutput="raw_values"),
    })

    summary = {
        "loss": loss,
        "mean_mae": float(per_target["MAE"].mean()),
        "mean_rmse": float(per_target["RMSE"].mean()),
        "mean_r2": float(per_target["R2"].mean()),
        "per_target": per_target,
        "y_true": y_true_real,
        "y_pred": y_pred_real,
    }
    return summary


def run_graphgps(df_features: pd.DataFrame, df_targets: pd.DataFrame, targets: list[str], id_col: str = "vk_id"):
    print_header("1) GraphGPS на локальном файле")

    merged = df_features.merge(df_targets, on=id_col, how="inner")
    edges = build_knn_edges(df_features, id_col=id_col, k=K_NEIGHBORS)

    print(f"Объектов: {len(merged)}")
    print(f"Признаков после препроцессинга: {len(df_features.columns) - 1}")
    print(f"Рёбер в kNN-графе: {len(edges)}")

    graph = PsychologyGraph(merged, edges, targets=targets, id_col=id_col)
    G = graph.build_graph()
    data = graph.to_pyg_data(G, add_pe=True, pe_dim=PE_DIM)
    data = add_masks(data, seed=RANDOM_SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GraphGPSRegressor(
        node_dim=data.x.shape[1],  # PE уже добавлен в x
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM,
        num_targets=len(targets),
        num_layers=NUM_LAYERS,
        pe_dim=0,
        heads=HEADS,
        dropout=DROPOUT,
        dropedge_p=DROPEDGE_P,
    ).to(device)

    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="min",
        factor=0.5,
        patience=10,
        min_lr=1e-5,
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, data, criterion, optimizer, device)
        val_metrics = evaluate_split(model, data, "val_mask", criterion, device, graph)
        val_loss = val_metrics["loss"]
        scheduler.step(val_loss)

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss={train_loss:.4f} | "
                f"Val Loss={val_loss:.4f} | "
                f"Val mean MAE={val_metrics['mean_mae']:.4f} | "
                f"Val mean R2={val_metrics['mean_r2']:.4f}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Ранняя остановка на эпохе {epoch}. Лучший val loss: {best_val_loss:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_split(model, data, "test_mask", criterion, device, graph)
    per_target = test_metrics["per_target"].copy()
    per_target.loc[len(per_target)] = {
        "Target": "MEAN",
        "MAE": per_target["MAE"].mean(),
        "RMSE": per_target["RMSE"].mean(),
        "R2": per_target["R2"].mean(),
    }

    print("\nGraphGPS — тестовые метрики:")
    print(per_target.to_string(index=False))
    return model, graph, data, per_target

In [35]:
def run_gradient_boosting(df_features: pd.DataFrame, df_targets: pd.DataFrame, targets: list[str], id_col: str = "vk_id"):
    print_header("2) Baseline: GradientBoostingRegressor")

    merged = df_features.merge(df_targets, on=id_col, how="inner")
    X = merged.drop(columns=[id_col] + targets)
    y = merged[targets]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    model = MultiOutputRegressor(
        GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_SEED,
        )
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics_df = pd.DataFrame({
        "Target": targets,
        "MAE": mean_absolute_error(y_test, y_pred, multioutput="raw_values"),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred, multioutput="raw_values")),
        "R2": r2_score(y_test, y_pred, multioutput="raw_values"),
    })
    metrics_df.loc[len(metrics_df)] = {
        "Target": "MEAN",
        "MAE": metrics_df["MAE"].mean(),
        "RMSE": metrics_df["RMSE"].mean(),
        "R2": metrics_df["R2"].mean(),
    }

    print(metrics_df.to_string(index=False))
    return model, metrics_df

In [36]:
def main() -> None:
    set_seed(RANDOM_SEED)

    print_header("Загрузка данных")
    df_features_raw, df_targets, original_feature_cols = load_single_file_dataset(
        DATA_FILE,
        id_col=ID_COL,
        targets=TARGETS,
    )
    print(f"Файл: {DATA_FILE}")
    print(f"Строк после очистки: {len(df_features_raw)}")
    print(f"Исходных признаков: {len(original_feature_cols)}")
    print(f"Таргеты: {TARGETS}")

    df_features = preprocess_features(df_features_raw, id_col=ID_COL)
    print(f"Признаков после препроцессинга: {len(df_features.columns) - 1}")

    _, graph_metrics = None, None
    try:
        _, _, _, graph_metrics = run_graphgps(df_features, df_targets, TARGETS, id_col=ID_COL)
    except Exception as e:
        print("\nGraphGPS не отработал:")
        print(e)

    _, gb_metrics = run_gradient_boosting(df_features, df_targets, TARGETS, id_col=ID_COL)

    print_header("Итог")
    if graph_metrics is not None:
        print("GraphGPS:")
        print(graph_metrics.to_string(index=False))
        print()
    print("GradientBoosting:")
    print(gb_metrics.to_string(index=False))


if __name__ == "__main__":
    main()



Загрузка данных
Файл прочитан как CSV: sep=',', encoding=utf-8
Файл: merged_file_new_2.csv
Строк после очистки: 124
Исходных признаков: 42
Таргеты: ['result/0/Экстраверсия–интроверсия', 'result/0/Привязанность–обособленность', 'result/0/Самоконтроль–импульсивность', 'result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость', 'result/0/Экспрессивность–практичность']
Удаляю полностью пустые признаки: 2
Признаков после препроцессинга: 685

1) GraphGPS на локальном файле
Объектов: 124
Признаков после препроцессинга: 685
Рёбер в kNN-графе: 755


/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_37976/4010895517.py:39: ParserWarning: Skipping line 126: unexpected end of data

  df = pd.read_csv(


Epoch 001 | Train Loss=0.3428 | Val Loss=0.0986 | Val mean MAE=15.1733 | Val mean R2=-1.5987
Epoch 010 | Train Loss=0.1056 | Val Loss=0.0582 | Val mean MAE=10.9597 | Val mean R2=-0.4941
Epoch 020 | Train Loss=0.0731 | Val Loss=0.0422 | Val mean MAE=9.3757 | Val mean R2=-0.0871
Epoch 030 | Train Loss=0.0616 | Val Loss=0.0407 | Val mean MAE=9.3624 | Val mean R2=-0.0568
Epoch 040 | Train Loss=0.0614 | Val Loss=0.0396 | Val mean MAE=9.2148 | Val mean R2=-0.0429
Epoch 050 | Train Loss=0.0659 | Val Loss=0.0404 | Val mean MAE=9.3188 | Val mean R2=-0.0503
Epoch 060 | Train Loss=0.0599 | Val Loss=0.0402 | Val mean MAE=9.2686 | Val mean R2=-0.0554
Epoch 070 | Train Loss=0.0607 | Val Loss=0.0398 | Val mean MAE=9.2670 | Val mean R2=-0.0466
Ранняя остановка на эпохе 73. Лучший val loss: 0.0392

GraphGPS — тестовые метрики:
                                                          Target       MAE      RMSE        R2
                               result/0/Экстраверсия–интроверсия 10.805290 13.27444